# 🐉 Graph BDH BabyLM: Training on TinyStories & Neuromorphic Energy Simulation

This notebook trains **Graph BDH** (Biologically-inspired Dual Hebbian with explicit local graph dynamics, sparse linear circuits, and integrate-and-fire spiking primitives) on **`karpathy/tinystories-gpt4-clean`**.

Following training, it runs a **neuromorphic hardware energy simulation** (modeling an Intel Loihi 2-class event-driven chip) to measure:
1. **Spike activation sparsity** (% of quiescent neurons)
2. **Synaptic Operations (SynOps)** vs. dense GPU MACs
3. **Energy consumption per token** in nanojoules ($nJ$) and microjoules ($\mu J$)
4. **Neuromorphic efficiency factor** ($10\times - 100\times$ lower energy than GPUs)

## 1. Setup & Dependencies
Ensure GPU is selected: `Runtime -> Change runtime type -> Hardware accelerator -> T4 or A100 GPU`.

In [ ]:
!pip install -q datasets transformers torch

## 2. Load Repository & Set Python Path
Clones `bdh-carbon` if running in a remote Colab environment and adds it to `sys.path`.

In [ ]:
import os
import sys

# If running on Google Colab remote runtime, clone or pull the repository
if not os.path.exists("graph_bdh.py"):
    if not os.path.exists("bdh-carbon"):
        !git clone https://github.com/brrryry/bdh-carbon.git
    if os.path.exists("bdh-carbon"):
        os.chdir("bdh-carbon")
else:
    # If already cloned, ensure we have the latest updates
    !git pull origin main 2>/dev/null || true

# Ensure the current folder is in Python's search path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
print("Working directory:", os.getcwd())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# Verify graph_bdh can be imported
import graph_bdh
print("Successfully imported graph_bdh!")

## 3. Train BabyLM on `karpathy/tinystories-gpt4-clean`

In [ ]:
# Run the BabyLM training pipeline
!python train_tinystories.py \
    --n_neurons 512 \
    --n_embd 64 \
    --n_layer 4 \
    --threshold_mode spiking_binary \
    --synaptic_topology hub_sparse \
    --batch_size 16 \
    --block_size 128 \
    --max_steps 1000 \
    --learning_rate 5e-4 \
    --eval_freq 100

## 4. Neuromorphic Hardware Energy Profiling
Simulate event-driven inference on Intel Loihi 2 neuromorphic specs.

In [ ]:
import torch
from graph_bdh import GraphBDH, GraphBDHConfig
from neuromorphic_profiler import NeuromorphicProfiler, NeuromorphicHardwareSpecs
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Allowlist GraphBDHConfig for PyTorch 2.6+ weights_only unpickler
try:
    torch.serialization.add_safe_globals([GraphBDHConfig])
except Exception:
    pass

# Load trained model checkpoint safely
try:
    checkpoint = torch.load("./checkpoints/graph_bdh_babylm.pt", map_location=device)
except Exception:
    checkpoint = torch.load("./checkpoints/graph_bdh_babylm.pt", map_location=device, weights_only=False)

raw_config = checkpoint["config"]
config = GraphBDHConfig(**raw_config) if isinstance(raw_config, dict) else raw_config
model = GraphBDH(config).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Configure Neuromorphic specs (Loihi 2: 3.5 pJ/SynOp)
specs = NeuromorphicHardwareSpecs(
    name="Intel-Loihi-2-Class",
    energy_per_synop_pj=3.5,
    energy_per_neuron_update_pj=40.0,
    gpu_energy_per_mac_pj=35.0,
)

profiler = NeuromorphicProfiler(model, specs=specs)
prompt = "Once upon a time, a little bird wanted to sing"
input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

generated, profiles = profiler.profile_generation(input_ids, max_new_tokens=50, temperature=0.8, top_k=5)
profiler.print_summary(profiles)

print("\n--- Generated Text ---")
print(tokenizer.decode(generated[0].tolist()))